# Activity 5.01: Clustering Sales Data Using K-Means

In this activity we use the **Sales Transactions Dataset Weekly** (UCI Machine Learning Repository). It contains the weekly sales of 811 products over one year (52 weeks), and no other information about the products.

**Goal:** use the k-means clustering algorithm to find products with similar sales trends, and experiment with the number of clusters to find a good value.

## Step 1 - Open a new Jupyter Notebook
This notebook is the new Jupyter Notebook file for the activity.

## Step 2 - Load the dataset and inspect the data
We import pandas and load the CSV file from the textbook's GitHub repository into a DataFrame. Then we check its size, look at the first rows and make sure there are no missing values.

In [1]:
# pandas loads and handles the dataset as a table (DataFrame)
import pandas as pd

# URL of the raw dataset in the textbook's GitHub repository
file_url = 'https://raw.githubusercontent.com/'\
           'PacktWorkshops/'\
           'The-Applied-Artificial-Intelligence-Workshop/'\
           'master/Datasets/'\
           'Sales_Transactions_Dataset_Weekly.csv'

# Load the CSV file into a DataFrame
df = pd.read_csv(file_url)

# Size of the table (rows = products, columns = variables)
print("Dataset shape (rows, columns):", df.shape)

# First five rows of the first 8 columns (the table is very wide)
print(df.iloc[:, :8].head())

Dataset shape (rows, columns): (811, 107)
  Product_Code  W0  W1  W2  W3  W4  W5  W6
0           P1  11  12  10   8  13  12  14
1           P2   7   6   3   2   7   1   6
2           P3   7  11   8   9  10   8   7
3           P4  12   8  13   5   9   6   9
4           P5   8   5  13  11   6   7   9


In [2]:
# Count the missing (empty) values in the whole table
print("Missing values in the dataset:", df.isnull().sum().sum())

# Show the names of the columns by group, to understand the layout
print("First column:", df.columns[0])
print("Weekly sales columns:", df.columns[1], "...", df.columns[52])
print("MIN / MAX columns:", df.columns[53], "and", df.columns[54])
print("Normalized columns:", df.columns[55], "...",
      df.columns[-1])

Missing values in the dataset: 0
First column: Product_Code
Weekly sales columns: W0 ... W51
MIN / MAX columns: MIN and MAX
Normalized columns: Normalized 0 ... Normalized 51


### What the columns mean
The dataset has 811 rows (one per product) and 107 columns:

* `Product_Code` - the product identifier (1 column).
* `W0` to `W51` - the sales quantity of each week (52 columns). The weeks are numbered 0 to 51.
* `MIN` and `MAX` - the lowest and highest weekly sales of the product (2 columns).
* `Normalized 0` to `Normalized 51` - the same weekly sales rescaled so that products with very different sales volumes can be compared by their *trend* (52 columns).

The normalized columns are the better choice for clustering, because they let k-means group products by the **shape** of their sales over the year instead of by how much they sell. There are no missing values.

## Step 3 - Create a new DataFrame without the unnecessary columns
The first 55 columns (`Product_Code`, `W0`-`W51`, `MIN`, `MAX`) are not needed for clustering. We remove them with pandas `drop()`. Setting `inplace=False` returns a **new** DataFrame (`df2`) and keeps the original `df` unchanged, because we need `df` again in Step 6.

In [3]:
# drop() removes the first 55 columns (positions 0 to 54).
# axis=1 means we are removing columns, not rows.
# inplace=False keeps the original df unchanged.
df2 = df.drop(df.iloc[:, 0:55], inplace=False, axis=1)

# Check the result: only the 52 normalized columns should remain
print("df2 shape (rows, columns):", df2.shape)
print("First column:", df2.columns[0])
print("Last column: ", df2.columns[-1])
print("Original df shape (unchanged):", df.shape)
print(df2.iloc[:, :5].head())

df2 shape (rows, columns): (811, 52)
First column: Normalized 0
Last column:  Normalized 51
Original df shape (unchanged): (811, 107)
   Normalized 0  Normalized 1  Normalized 2  Normalized 3  Normalized 4
0          0.44          0.50          0.39          0.28          0.56
1          0.70          0.60          0.30          0.20          0.70
2          0.36          0.73          0.45          0.55          0.64
3          0.59          0.35          0.65          0.18          0.41
4          0.33          0.13          0.67          0.53          0.20


## Step 4 - Create the k-means clustering model with 8 clusters
We build a `KMeans` model with `n_clusters=8` and `random_state=8`. The `random_state` makes the result reproducible, because k-means starts from random cluster centers.

`n_init=10` means the algorithm is run 10 times with different starting centers and the best run is kept. We set it explicitly so the result does not depend on the scikit-learn version (the default changed in newer versions). Then `fit()` trains the model on `df2`.

In [4]:
# KMeans is the k-means clustering algorithm in scikit-learn
from sklearn.cluster import KMeans

# 8 clusters, fixed random_state for reproducible results,
# 10 runs with different starting centers (best one is kept)
k_means_model = KMeans(n_clusters=8, random_state=8, n_init=10)

# Train the model: find the 8 cluster centers
k_means_model.fit(df2)

print(k_means_model)

KMeans(n_init=10, random_state=8)


## Step 5 - Retrieve the labels from the clustering model
After training, `labels_` holds the cluster number (0 to 7) assigned to each product. Row *i* of the dataset belongs to the cluster `labels[i]`. We also count how many products fall in each cluster.

In [5]:
# Cluster number (0-7) assigned to every product
labels = k_means_model.labels_

print("Number of labels:", len(labels))
print("Labels:")
print(labels)

# How many products are in each cluster
print("\nNumber of products per cluster:")
print(pd.Series(labels).value_counts().sort_index())

Number of labels: 811
Labels:
[7 7 3 7 7 0 7 3 3 3 7 3 4 0 4 4 4 4 4 3 4 4 4 4 4 0 4 4 7 4 4 4 3 4 4 7 4
 7 7 4 4 4 4 4 7 4 7 4 4 7 4 4 3 4 4 4 4 3 4 4 4 4 4 4 3 4 4 7 4 4 4 4 4 7
 7 4 3 4 7 4 3 7 7 7 4 4 4 4 4 4 3 4 4 0 3 7 4 3 7 5 4 4 4 7 7 0 3 5 7 7 5
 7 4 4 4 3 7 7 4 4 7 7 3 7 7 7 7 4 4 4 4 7 4 7 4 4 4 7 4 4 4 4 4 3 7 3 4 7
 4 3 7 3 7 3 7 3 3 7 7 3 7 7 7 3 4 7 4 7 7 4 4 4 4 4 4 4 7 4 4 4 4 4 4 4 4
 4 4 4 7 4 4 4 7 4 7 4 3 0 3 0 7 0 3 0 0 0 0 0 0 3 3 6 6 2 2 6 1 6 6 1 6 6
 2 1 1 2 2 2 1 2 6 2 2 2 6 2 2 1 2 6 2 6 2 2 2 2 2 6 2 2 2 6 2 2 6 2 5 2 2
 2 3 0 5 3 0 3 3 0 0 0 7 2 2 6 2 2 1 6 2 6 5 5 2 4 0 0 1 5 1 6 7 7 7 7 0 0
 7 3 3 3 7 3 4 7 7 7 5 7 4 4 7 7 7 3 7 7 3 7 3 3 7 3 3 7 7 7 7 3 3 0 7 3 4
 7 7 0 0 5 5 1 3 0 5 0 5 6 1 1 6 2 2 6 2 2 0 7 5 0 5 0 5 1 0 7 7 0 0 0 0 5
 5 5 5 6 5 1 5 2 6 2 2 2 6 2 0 0 0 7 5 0 5 5 5 0 0 0 0 0 7 7 0 0 0 0 3 0 0
 0 0 7 0 0 3 5 5 2 6 2 2 1 5 6 2 6 2 6 0 0 0 0 6 5 3 5 3 1 1 5 5 1 2 2 2 2
 6 6 6 6 2 6 2 2 1 6 2 1 6 1 2 1 6 2 6 2 2 2 2 2 2 2 2 6 0 5 5 5 6 6 5

## Step 6 - Keep only the W columns and add the labels
Now we go back to the first DataFrame, `df`, and keep only the weekly sales columns `W0` to `W51` (this removes `Product_Code`, `MIN`, `MAX` and the normalized columns). Then we add the cluster labels as a new column called `label`.

We use the real sales numbers (not the normalized ones) here because the business question is about sales: how much do the products of each cluster sell?

In [6]:
# Keep only the weekly sales columns W0 ... W51
df = df.loc[:, 'W0':'W51'].copy()

# Add the cluster label of each product as a new column
df['label'] = labels

print("Shape:", df.shape)
print(df[['W0', 'W1', 'W2', 'W51', 'label']].head())

Shape: (811, 53)
   W0  W1  W2  W51  label
0  11  12  10   10      7
1   7   6   3    0      7
2   7  11   8    7      3
3  12   8  13    8      7
4   8   5  13    9      7


## Step 7 - Aggregate the data to get the yearly average sales of each cluster
We use the pandas `groupby()` function to summarize each cluster:

1. `groupby('label').sum()` adds up the sales of every week for the products in each cluster (`df_agg`).
2. `groupby('label').count()` on one column (`W0`) counts the products per cluster (`count_product`).
3. Summing all weekly columns of `df_agg` gives the total sales of the cluster (`total_sales`).
4. `total_sales / count_product` gives the **yearly average sales of one product** in that cluster.
5. We sort the clusters from the highest to the lowest yearly average sales.

In [7]:
# Total sales of each week for every cluster
df_agg = df.groupby('label').sum()

# Number of products per cluster (counting any single column works)
df_final = df[['label', 'W0']].groupby('label').count()
df_final = df_final.rename(columns={'W0': 'count_product'})

# Total sales of the whole year for every cluster
df_final['total_sales'] = df_agg.sum(axis=1)

# Yearly average sales of one product in the cluster
df_final['yearly_average_sales'] = \
    df_final['total_sales'] / df_final['count_product']

# Best-selling clusters first
df_final = df_final.sort_values(by='yearly_average_sales',
                                ascending=False)

print(df_final)

       count_product  total_sales  yearly_average_sales
label                                                  
4                125       180199           1441.592000
7                119        71217            598.462185
3                 98        50656            516.897959
0                127        57363            451.677165
5                 89        11493            129.134831
1                 54         1704             31.555556
6                 84         1895             22.559524
2                115          760              6.608696


## Extra - Experiment with the number of clusters
The activity asks us to experiment with the number of clusters. A common tool is the **elbow method**: we train k-means for several values of *k* and look at the **inertia** (the sum of squared distances between each product and its cluster center). Inertia always decreases as *k* grows, so we look for the point where the improvement becomes small (the "elbow").

In [8]:
# Inertia of k-means for k = 2 ... 12 (same settings as before)
# Use the normalized data (df2) again
inertias = {}
for k in range(2, 13):
    model = KMeans(n_clusters=k, random_state=8, n_init=10)
    model.fit(df2)
    inertias[k] = model.inertia_

# Print k, inertia and the drop compared with the previous k
print("k   inertia   drop from previous k")
previous = None
for k, value in inertias.items():
    if previous is None:
        print(k, "  ", round(value, 1))
    else:
        print(k, "  ", round(value, 1), "  ", round(previous - value, 1))
    previous = value

k   inertia   drop from previous k
2    2402.7
3    2271.5    131.1
4    2202.3    69.2
5    2166.6    35.7
6    2133.5    33.1
7    2115.8    17.8
8    2088.8    27.0
9    2079.0    9.8
10    2054.9    24.1
11    2054.3    0.6
12    2038.5    15.8


### Conclusion - Activity 5.01
K-means with 8 clusters grouped the 811 products by the shape of their yearly sales, and the clusters turned out to be very different in business value:

* **Best-selling cluster (label 4):** 125 products (about 15% of all products) with a yearly average of about **1,442 units per product**. Together they sold 180,199 units, which is about 48% of all units sold. Clusters 7, 3 and 0 follow with yearly averages of about 598, 517 and 452 units.
* **Worst-selling cluster (label 2):** 115 products that sold only about **6.6 units per product** in the whole year (760 units in total). Clusters 6 and 1 are also very weak, with about 23 and 32 units per product.
* **Business value:** without using any product name or description, k-means separated the strong sellers from the weak ones automatically. A company could protect the stock of the top cluster and review whether the products in the bottom clusters are worth keeping.

**Choosing the number of clusters:** the inertia table shows large improvements up to about k = 4 (drops of 131 and 69), then smaller and less regular improvements (for example 35.7, 33.1 and 17.8). There is no single sharp elbow, so several values of k are reasonable. We kept k = 8 as the activity asks, because it gives a more detailed ranking of the products than k = 4 or k = 5.

**Note:** k-means depends on random starting points and on the scikit-learn version, so the cluster numbers and sizes can differ from the figure printed in the textbook. The overall pattern (a few high-selling clusters and several very low-selling ones) is what matters.